In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore') # To ignore all warnings
warnings.filterwarnings('ignore', category=UserWarning) # To ignore specific category

np.random.seed(42)  # for reproducibility

n_entities = 1000
n_transactions = 5000

# Create synthetic entity IDs
entities = [f"User_{i}" for i in range(n_entities)]

# Randomly generate transactions
df_large = pd.DataFrame({
    'sender': np.random.choice(entities, n_transactions),
    'receiver': np.random.choice(entities, n_transactions),
    'amount': np.round(np.random.exponential(scale=1000, size=n_transactions), 2),
    'timestamp': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(0, 180, size=n_transactions), unit='D')
})

# Remove self-transfers
df_large = df_large[df_large['sender'] != df_large['receiver']].reset_index(drop=True)

print(df_large.head())
print(f"Total transactions: {len(df_large)}")

df_large['day'] = df_large['timestamp'].dt.date
daily_graphs = {day: df_large[df_large['day'] == day] for day in df_large['day'].unique()}

display(daily_graphs)

     sender  receiver  amount  timestamp
0  User_102  User_973  224.98 2023-06-09
1  User_435  User_631  308.17 2023-01-21
2  User_860  User_893  186.04 2023-06-17
3  User_270  User_566  100.53 2023-01-27
4  User_106   User_26  543.24 2023-01-02
Total transactions: 4993


{datetime.date(2023, 6, 9):         sender  receiver   amount  timestamp         day
 0     User_102  User_973   224.98 2023-06-09  2023-06-09
 35    User_252  User_659  1107.92 2023-06-09  2023-06-09
 290   User_384  User_227   189.98 2023-06-09  2023-06-09
 964   User_731  User_552   688.19 2023-06-09  2023-06-09
 1155  User_863   User_62  1492.02 2023-06-09  2023-06-09
 1310  User_898  User_508  1327.13 2023-06-09  2023-06-09
 1563  User_904  User_684   727.75 2023-06-09  2023-06-09
 1594  User_201  User_574    12.66 2023-06-09  2023-06-09
 1673  User_768  User_694  1165.90 2023-06-09  2023-06-09
 2107  User_301  User_472   842.75 2023-06-09  2023-06-09
 2109   User_10  User_271   824.59 2023-06-09  2023-06-09
 2292  User_413  User_359   725.95 2023-06-09  2023-06-09
 2362  User_796   User_90    84.53 2023-06-09  2023-06-09
 2427  User_692  User_673  2844.15 2023-06-09  2023-06-09
 2555   User_21  User_189  2722.20 2023-06-09  2023-06-09
 2718  User_275  User_977   636.93 2023-06-09

In [6]:
# build graph

import networkx as nx

G = nx.from_pandas_edgelist(
    df_large,
    source='sender',
    target='receiver',
    edge_attr=['amount', 'timestamp'],
    create_using=nx.DiGraph()
)
print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")


# Note:
# Node	A point or entity in the network
# Edge	A connection or relationship between nodes
# Node: A bank account or customer
# Edge: A wire transfer between accounts
# Node: A user
# Edge: A friendship or message

Graph has 1000 nodes and 4969 edges


In [7]:
# Engineer Features and Train a Model

# Add a circular fraud ring
ring_entities = [f"Fraud_{i}" for i in range(10)]
ring_tx = []

for i in range(len(ring_entities)):
    sender = ring_entities[i]
    receiver = ring_entities[(i + 1) % len(ring_entities)]
    ring_tx.append({
        'sender': sender,
        'receiver': receiver,
        'amount': np.random.randint(5000, 20000),
        'timestamp': pd.to_datetime('2023-03-01')
    })

# Inject into main dataset
df_large = pd.concat([df_large, pd.DataFrame(ring_tx)], ignore_index=True)

display(df_large)

,sender,receiver,amount,timestamp,day
0,User_102,User_973,224.98,2023-06-09,2023-06-09
1,User_435,User_631,308.17,2023-01-21,2023-01-21
2,User_860,User_893,186.04,2023-06-17,2023-06-17
3,User_270,User_566,100.53,2023-01-27,2023-01-27
4,User_106,User_26,543.24,2023-01-02,2023-01-02
...,...,...,...,...,...
4998,Fraud_5,Fraud_6,14007.00,2023-03-01,NaN
4999,Fraud_6,Fraud_7,7326.00,2023-03-01,NaN
5000,Fraud_7,Fraud_8,7330.00,2023-03-01,NaN
5001,Fraud_8,Fraud_9,12646.00,2023-03-01,NaN


In [8]:
# Create Pyvis Graph

# Inject synthetic fraud ring
fraud_ring_ids = [f"User_{900 + i}" for i in range(10)]
fraud_transactions = []

for i in range(len(fraud_ring_ids)):
    fraud_transactions.append({
        'sender': fraud_ring_ids[i],
        'receiver': fraud_ring_ids[(i + 1) % len(fraud_ring_ids)],
        'amount': 9999.99,
        'timestamp': pd.Timestamp('2023-03-01')
    })

# Add fraud to the main DataFrame
df_fraud = pd.DataFrame(fraud_transactions)
df_large = pd.concat([df_large, df_fraud], ignore_index=True).reset_index(drop=True)

# Flag fraud
df_large['is_fraud'] = False
df_large.loc[df_large.index[-len(fraud_transactions):], 'is_fraud'] = True

display(df_large.head())

,sender,receiver,amount,timestamp,day,is_fraud
0,User_102,User_973,224.98,2023-06-09,2023-06-09,False
1,User_435,User_631,308.17,2023-01-21,2023-01-21,False
2,User_860,User_893,186.04,2023-06-17,2023-06-17,False
3,User_270,User_566,100.53,2023-01-27,2023-01-27,False
4,User_106,User_26,543.24,2023-01-02,2023-01-02,False


In [12]:
!pip install pyvis -q
import pandas as pd
import numpy as np
from pyvis.network import Network
from IPython.display import HTML

# Step 1: Create large transaction dataset
np.random.seed(42)
n_entities = 1000
n_transactions = 5000
entities = [f"User_{i}" for i in range(n_entities)]

df_large = pd.DataFrame({
    'sender': np.random.choice(entities, n_transactions),
    'receiver': np.random.choice(entities, n_transactions),
    'amount': np.round(np.random.exponential(scale=1000, size=n_transactions), 2),
    'timestamp': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(0, 180, size=n_transactions), unit='D')
})
df_large = df_large[df_large['sender'] != df_large['receiver']].reset_index(drop=True)

# Step 2: Inject synthetic fraud ring
fraud_ring_ids = [f"User_{900 + i}" for i in range(10)]
fraud_transactions = [{
    'sender': fraud_ring_ids[i],
    'receiver': fraud_ring_ids[(i + 1) % len(fraud_ring_ids)],
    'amount': 9999.99,
    'timestamp': pd.Timestamp('2023-03-01')
} for i in range(len(fraud_ring_ids))]

df_fraud = pd.DataFrame(fraud_transactions)
df_large = pd.concat([df_large, df_fraud], ignore_index=True)
df_large['is_fraud'] = False
df_large.loc[df_large.index[-len(fraud_transactions):], 'is_fraud'] = True

# Step 3: Sample for visualization
df_sample = pd.concat([
    df_large[df_large['is_fraud']],
    df_large[~df_large['is_fraud']].sample(200, random_state=42)
])

# Step 4: Create interactive graph
net = Network(height="700px", width="100%", directed=True, notebook=True, cdn_resources='remote')
for _, row in df_sample.iterrows():
    edge_color = "red" if row['is_fraud'] else "green"
    net.add_node(row['sender'], label=row['sender'], color='orange')
    net.add_node(row['receiver'], label=row['receiver'], color='orange')
    net.add_edge(row['sender'], row['receiver'], color=edge_color, title=f"${row['amount']}")

# Step 5: Save and Display manually to avoid connection errors
net.save_graph("fraud_detection_network.html")
HTML(filename="fraud_detection_network.html")

In [15]:
import pandas as pd
import numpy as np
from pyvis.network import Network
from IPython.display import HTML

# Step 1: Create large transaction dataset
np.random.seed(42)
n_entities = 1000
n_transactions = 5000
entities = [f"User_{i}" for i in range(n_entities)]

df_large = pd.DataFrame({
    'sender': np.random.choice(entities, n_transactions),
    'receiver': np.random.choice(entities, n_transactions),
    'amount': np.round(np.random.exponential(scale=1000, size=n_transactions), 2),
    'timestamp': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(0, 180, size=n_transactions), unit='D')
})
df_large = df_large[df_large['sender'] != df_large['receiver']].reset_index(drop=True)

# Step 2: Inject synthetic fraud ring
fraud_ring_ids = [f"User_{900 + i}" for i in range(10)]
fraud_transactions = [{
    'sender': fraud_ring_ids[i],
    'receiver': fraud_ring_ids[(i + 1) % len(fraud_ring_ids)],
    'amount': 9999.99,
    'timestamp': pd.Timestamp('2023-03-01')
} for i in range(len(fraud_ring_ids))]

df_fraud = pd.DataFrame(fraud_transactions)
df_large = pd.concat([df_large, df_fraud], ignore_index=True)
df_large['is_fraud'] = False
df_large.loc[df_large.index[-len(fraud_transactions):], 'is_fraud'] = True

# Step 3: Sample for visualization
df_sample = pd.concat([
    df_large[df_large['is_fraud']],
    df_large[~df_large['is_fraud']].sample(200, random_state=42)
])

# Step 4: Create interactive graph
net = Network(height="700px", width="100%", directed=True, notebook=True, cdn_resources='remote')
# Track nodes we've already added
added_nodes = set()

for _, row in df_sample.iterrows():
    edge_color = "red" if row['is_fraud'] else "green"

    for node in [row['sender'], row['receiver']]:
        if node not in added_nodes:
            # Check if this node appears in any fraud transaction
            is_fraud_node = df_sample[
                ((df_sample['sender'] == node) | (df_sample['receiver'] == node)) &
                (df_sample['is_fraud'])
            ].any().any()

            node_color = "red" if is_fraud_node else "green"
            net.add_node(node, label=node, color=node_color)
            added_nodes.add(node)

    net.add_edge(row['sender'], row['receiver'], color=edge_color, title=f"${row['amount']}")

# Step 5: Save and Display
net.save_graph("fraud_detection_network.html")
HTML(filename="fraud_detection_network.html")

In [17]:
import pandas as pd
import numpy as np
from pyvis.network import Network
from IPython.display import HTML

# Step 1: Create large transaction dataset
np.random.seed(42)
n_entities = 1000
n_transactions = 5000
entities = [f"User_{i}" for i in range(n_entities)]

df_large = pd.DataFrame({
    'sender': np.random.choice(entities, n_transactions),
    'receiver': np.random.choice(entities, n_transactions),
    'amount': np.round(np.random.exponential(scale=1000, size=n_transactions), 2),
    'timestamp': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(0, 180, size=n_transactions), unit='D')
})
df_large = df_large[df_large['sender'] != df_large['receiver']].reset_index(drop=True)

# Step 2: Inject synthetic fraud ring
fraud_ring_ids = [f"User_{900 + i}" for i in range(10)]
fraud_transactions = [{
    'sender': fraud_ring_ids[i],
    'receiver': fraud_ring_ids[(i + 1) % len(fraud_ring_ids)],
    'amount': 9999.99,
    'timestamp': pd.Timestamp('2023-03-01')
} for i in range(len(fraud_ring_ids))]

df_fraud = pd.DataFrame(fraud_transactions)
df_large = pd.concat([df_large, df_fraud], ignore_index=True)
df_large['is_fraud'] = False
df_large.loc[df_large.index[-len(fraud_transactions):], 'is_fraud'] = True

# Step 3: Sample for visualization
df_sample = pd.concat([
    df_large[df_large['is_fraud']],
    df_large[~df_large['is_fraud']].sample(200, random_state=42)
])

# Sum of amounts sent + received per user in the sample
node_amounts = (
    df_sample.groupby('sender')['amount'].sum().add(
    df_sample.groupby('receiver')['amount'].sum(), fill_value=0)
)

# Step 4: Create interactive graph
net = Network(height="700px", width="100%", directed=True, notebook=True, cdn_resources='remote')
added_nodes = set()

for _, row in df_sample.iterrows():
    edge_color = "red" if row['is_fraud'] else "green"

    for node in [row['sender'], row['receiver']]:
        if node not in added_nodes:
            is_fraud_node = df_sample[
                ((df_sample['sender'] == node) | (df_sample['receiver'] == node)) &
                (df_sample['is_fraud'])
            ].any().any()

            node_color = "red" if is_fraud_node else "#FFA500" # Orange for normal

            total_amt = node_amounts.get(node, 0)
            node_label = f"{node}\n${total_amt:,.2f}"

            net.add_node(node, label=node_label, color=node_color)
            added_nodes.add(node)

    net.add_edge(row['sender'], row['receiver'], color=edge_color, title=f"${row['amount']}")

# Step 5: Save and Show
net.save_graph("fraud_detection_network.html")
HTML(filename="fraud_detection_network.html")